Data Cleaning

In [ ]:
import pandas as pd
import numpy as np

# Load CSV file
df = pd.read_csv(r'C:\Users\Numan\OneDrive\York University\2) Business Applications of AI I (MMAI 5040U)\Group Project\Major_Crime_Indicators_Open_Data.csv', encoding='utf-8')

# Change Column Types
df = df.astype({
    'X': 'float',
    'Y': 'float',
    'OBJECTID': 'int64',
    'EVENT_UNIQUE_ID': 'str',
    'REPORT_MONTH': 'str',
    'REPORT_DAY': 'Int64',
    'REPORT_DOY': 'Int64',
    'REPORT_DOW': 'str',
    'REPORT_HOUR': 'Int64',
    'OCC_MONTH': 'str',
    'OCC_DAY': 'Int64',
    'OCC_DOY': 'Int64',
    'OCC_DOW': 'str',
    'OCC_HOUR': 'Int64',
    'DIVISION': 'str',
    'LOCATION_TYPE': 'str',
    'PREMISES_TYPE': 'str',
    'UCR_CODE': 'Int64',
    'UCR_EXT': 'Int64',
    'OFFENCE': 'str',
    'MCI_CATEGORY': 'str',
    'HOOD_158': 'str',
    'NEIGHBOURHOOD_158': 'str',
    'HOOD_140': 'str',
    'NEIGHBOURHOOD_140': 'str',
    'LONG_WGS84': 'float',
    'LAT_WGS84': 'float'
})

# Change Type to Date for specific columns
df['REPORT_DATE'] = pd.to_datetime(df['REPORT_DATE'], errors='coerce').dt.date
df['OCC_DATE'] = pd.to_datetime(df['OCC_DATE'], errors='coerce').dt.date

# Remove unnecessary columns
df.drop(columns=[
    'X', 'Y', 'OBJECTID', 'REPORT_DOY', 'OCC_YEAR', 'OCC_MONTH', 'OCC_DOY', 'OCC_DOW',
    'UCR_CODE', 'UCR_EXT', 'HOOD_140', 'NEIGHBOURHOOD_140', 'REPORT_YEAR', 'REPORT_MONTH',
    'REPORT_DAY', 'REPORT_DOW', 'OCC_DAY'
], inplace=True)

# Rename Columns
df.rename(columns={
    'EVENT_UNIQUE_ID': 'Event ID',
    'REPORT_DATE': 'Reported Date',
    'DIVISION': 'Division',
    'LOCATION_TYPE': 'Location Type',
    'PREMISES_TYPE': 'Location Group',
    'OFFENCE': 'Offence',
    'MCI_CATEGORY': 'Offence Group',
    'HOOD_158': 'Neighbourhood #',
    'NEIGHBOURHOOD_158': 'Neighbourhood',
    'LONG_WGS84': 'Long',
    'LAT_WGS84': 'Lat',
    'OCC_HOUR': 'Occurance Hour',
    'OCC_DATE': 'Occurance Date',
    'REPORT_HOUR': 'Reported Hour'
}, inplace=True)

# Sort Rows by 'Event ID'
df.sort_values(by='Event ID', ascending=True, inplace=True)

# Remove Duplicates
df.drop_duplicates(inplace=True)

# Save the cleaned DataFrame to a new CSV file with "_Cleaned" appended to the original file name
output_file_path = r'C:\Users\Numan\OneDrive\York University\2) Business Applications of AI I (MMAI 5040U)\Group Project\Major_Crime_Indicators_Open_Data_Cleaned.csv'
df.to_csv(output_file_path, index=False)

# Print success message and return the output path
print("File successfully saved to:")
output_file_path


Merge with Population data, Calculate per Capita Crime Rates per Neighborhood

In [ ]:
import pandas as pd

# Load the source data
source_path = 'path/to/raw/data/Major_Crime_Indicators_Open_Data_Cleaned.csv'
df = pd.read_csv(source_path)

# Remove duplicate rows based on "Event ID"
df = df.drop_duplicates(subset=['Event ID'])

# Add a column for "Occurance Year"
df['Occurance Year'] = pd.to_datetime(df['Occurance Date']).dt.year

# Filter data for the years after 2013
df = df[df['Occurance Year'] > 2013]

# Group rows by "Neighbourhood" and "Occurance Year" and count occurrences
df_grouped = df.groupby(['Neighbourhood', 'Occurance Year']).size().reset_index(name='Crime Occurance')

# Pivot the "Occurance Year" column
df_pivot = df_grouped.pivot(index='Neighbourhood', columns='Occurance Year', values='Crime Occurance').fillna(0)

# Remove the year 2024 if it exists in columns
df_pivot = df_pivot.drop(columns=[2024], errors='ignore')

# Add a column for total crime occurred from 2014 to 2023
df_pivot['Total Crime Occured (2014-2023)'] = df_pivot.loc[:, 2014:2023].sum(axis=1)

# Load neighbourhood population data
population_path = 'path/to/population_data/Neighoburhood_Populations.csv'
population_df = pd.read_csv(population_path)

# Merge with population data
df_merged = df_pivot.merge(population_df, on='Neighbourhood', how='left')

# Filter out rows with null populations
df_merged = df_merged[df_merged['Population'].notna()]

# Add CAGR of Crime (Absolute)
def calculate_cagr(start, end, periods):
    if start != 0:
        return (pow(end / start, 1 / periods) - 1)
    else:
        return None

df_merged['CAGR of Crime (Absolute)'] = df_merged.apply(lambda row: calculate_cagr(row[2014], row[2023], 2023 - 2014) if row[2014] != 0 else None, axis=1)

# Add total crime per capita
df_merged['Total Crime per Capita'] = df_merged['Total Crime Occured (2014-2023)'] / df_merged['Population']

# Add total crime per 1000 people
df_merged['Total Crime per 1000'] = df_merged['Total Crime per Capita'] * 1000

# Changing the data types to match the original intent
df_merged = df_merged.astype({
    'CAGR of Crime (Absolute)': 'float64',
    'Total Crime per Capita': 'float64',
    'Population': 'int64',
    'Total Crime Occured (2014-2023)': 'int64'
})

# Save the final dataframe
df_merged.to_csv('path/to/save/Crime_per_Neighbourhood.csv', index=False)